# Data Preparation

Der erste Schritt normalisiert die ROhdaten und reichert sie mit query-relevanten Daten an. Das Anreichern wird mit einem LLM durchgeführt. Die Daten prüfe ich nur stichprobenartig, sie gelten erstmal als so wahr.

- Rohdaten: Aus einem Vibecoding-Projekt.
- LLM: Mistral 

In [2]:
import os
import json
import pandas as pd
from tqdm import tqdm
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()

with open('../data/raw/products_raw.json', 'r') as f:
     products = json.load(f)

# Evaluation
products = products[:3]

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key)

def agent_request(system_promt, schema, content):
            
    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            "type": "json_object",
            "json_schema": schema
        }
    )

    return response

## Beschreibungen

Die Beschreibungen sollen so gegliedert sein, dass jeder Absatz ein Thema behandelt und das Produktname und Hersteller genannt wird. Werbliche Texte sollen entfernt werden. Die Antwort wird als JSON erwartet, es wird die dazu _response_format_ der API genutzt.

In [ ]:
with open('../data/promts/descs_agent.md', 'r') as f:
    descs_promt = f.read()

with open('../data/promts/descs_schema.json', 'r')as f:
    descs_schema = json.load(f)

for product in tqdm(products, total=len(products)):

    desc_response = agent_request(descs_promt, descs_schema, product['description'])
    product['desc_documents'] = json.loads(desc_response.choices[0].message.content)
    product['desc_usage'] = desc_response.usage.model_dump()


## Technische Daten

Bei den technischen Daten werden prinzipiell die selben Daten wie zur Beschreibung hinzugefügt, allerdings pro Objekt.

In [ ]:
with open('../data/promts/specs_agent.md', 'r') as f:
    specs_promt = f.read()

with open('../data/promts/specs_schema.json', 'r') as f:
    specs_schema = json.load(f)

for product in tqdm(products, total=len(products), desc="Products"):

    specs_serialaized = json.dumps(product['specs'])
    
    specs_response = agent_request(specs_promt, specs_schema, specs_serialaized)
    product['specs_documents'] = json.loads(specs_response.choices[0].message.content)
    product['specs_usage'] = specs_response.usage.model_dump()

Products: 100%|██████████| 3/3 [02:32<00:00, 50.99s/it]


## Speichern

In [36]:
with open("../data/processed/products_enriched.json", "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=2)

## Evaluation

Einmal nachsehen wie lange die Documents geworden sind und ob alle gefüllt wurden

In [12]:
with open('../data/processed/products_enriched.json', 'r', encoding='utf-8') as f:
    evaldata = json.load(f)

specs_chunks = []
descs_chunks = []
costs = []

for product in evaldata:

    costs.append({
        'id': product['id'],
        'type': 'descs',
        'prompt_tokens': product['desc_usage']['prompt_tokens'],
        'prompt_tokens': product['desc_usage']['completion_tokens'],
        'prompt_tokens': product['desc_usage']['total_tokens'],
    })

    costs.append({
        'id': product['id'],
        'type': 'specs',
        'prompt_tokens': product['specs_usage']['prompt_tokens'],
        'prompt_tokens': product['specs_usage']['completion_tokens'],
        'prompt_tokens': product['specs_usage']['total_tokens'],
    })

    for i, doc in enumerate(product.get('desc_document')):

        descs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': doc
        })

    for i, spec in enumerate(product.get('specs_document')):

        specs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': spec['natural_language_description']
        })

specs_df = pd.DataFrame(specs_chunks)

print(specs_df.info())
print(specs_df.head(5))
print(specs_df.columns)



# Prüfen wieviele Absätze pro Produkt (min, max, mean)
# Prüfen wieviele Specs pro Produkt (min, max, mean)

# Längen der Absätze (min, max,mean)
# Längen der Specs (min, max, mean)

# Stichproben

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      121 non-null    object
 1   num     121 non-null    object
 2   doc     121 non-null    object
dtypes: object(3)
memory usage: 3.0+ KB
None
                                             id num  \
0  Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank  00   
1  Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank  01   
2  Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank  02   
3  Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank  03   
4  Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank  04   

                                                 doc  
0  Der Kirsch LABO-288 PRO-ACTIVE hat die Außenma...  
1  Bei 90 Grad geöffneter Tür hat der Kirsch LABO...  
2  Die Innenmaße des Kirsch LABO-288 PRO-ACTIVE b...  
3  Der Kirsch LABO-288 PRO-ACTIVE hat einen Kühli...  
4  Die maximale Wärmeabgabe des Kirsch LABO-288 P...  
Index(['id